In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.data import HeteroData
from torch_geometric.nn import GATConv

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, roc_auc_score

c:\Users\Tanmai\.conda\envs\myenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_df = pd.read_csv("fraudTrain.csv")
test_df  = pd.read_csv("fraudTest.csv")

df = pd.concat([train_df, test_df]).sample(150000)  # 🔥 prevent memory crash
print(df.shape)

(150000, 23)


In [3]:
df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])

df['hour'] = df['trans_date_trans_time'].dt.hour
df['day']  = df['trans_date_trans_time'].dt.dayofweek

df['txn_during_night'] = df['hour'].apply(lambda x: 1 if x < 6 or x > 22 else 0)
df['is_weekend'] = df['day'].apply(lambda x: 1 if x >= 5 else 0)

df['distance'] = np.sqrt(
    (df['lat'] - df['merch_lat'])**2 +
    (df['long'] - df['merch_long'])**2
)

# Normalize
scaler = MinMaxScaler()
df[['amt','distance','hour','day']] = scaler.fit_transform(df[['amt','distance','hour','day']])

df.fillna(0, inplace=True)

In [4]:
fraud = df[df['is_fraud'] == 1]
normal = df[df['is_fraud'] == 0].sample(len(fraud) * 5)

df = pd.concat([fraud, normal]).sample(frac=1)
print(df['is_fraud'].value_counts())

is_fraud
0    3770
1     754
Name: count, dtype: int64


In [5]:
data = HeteroData()

customers = df['cc_num'].unique()
merchants = df['merchant'].unique()

cust_map = {c:i for i,c in enumerate(customers)}
merch_map = {m:i for i,m in enumerate(merchants)}

# Node features
cust_features = [[df[df['cc_num']==c].iloc[0]['lat'],
                  df[df['cc_num']==c].iloc[0]['long']] for c in customers]

merch_features = [[df[df['merchant']==m].iloc[0]['merch_lat'],
                   df[df['merchant']==m].iloc[0]['merch_long']] for m in merchants]

data['customer'].x = torch.tensor(cust_features, dtype=torch.float)
data['merchant'].x = torch.tensor(merch_features, dtype=torch.float)

# Edges
src, dst, labels = [], [], []

for _, row in df.iterrows():
    src.append(cust_map[row['cc_num']])
    dst.append(merch_map[row['merchant']])
    labels.append(row['is_fraud'])

edge_index = torch.tensor([src, dst])
y = torch.tensor(labels, dtype=torch.float)

data['customer','transacts','merchant'].edge_index = edge_index
data['customer','transacts','merchant'].y = y

In [6]:
num_edges = edge_index.shape[1]
perm = torch.randperm(num_edges)

train_size = int(0.8 * num_edges)

train_mask = torch.zeros(num_edges, dtype=torch.bool)
train_mask[perm[:train_size]] = True

test_mask = ~train_mask

In [7]:
class HybridGNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.gat1 = GATConv(2, 64, heads=2)
        self.gat2 = GATConv(128, 64)

        self.lin = nn.Linear(64, 32)

        # Autoencoder
        self.ae_enc = nn.Linear(32, 16)
        self.ae_dec = nn.Linear(16, 32)

    def forward(self, data):
        x_c = data['customer'].x
        x_m = data['merchant'].x
        edge_index = data['customer','transacts','merchant'].edge_index

        # Combine nodes
        x = torch.cat([x_c, x_m], dim=0)

        edge_index_shifted = edge_index.clone()
        edge_index_shifted[1] += x_c.shape[0]

        x = F.relu(self.gat1(x, edge_index_shifted))
        x = F.relu(self.gat2(x, edge_index_shifted))

        x = self.lin(x)

        z_c = x[:x_c.shape[0]]
        z_m = x[x_c.shape[0]:]

        src, dst = edge_index
        pred = torch.sigmoid(F.cosine_similarity(z_c[src], z_m[dst]))

        # Autoencoder
        z = torch.cat([z_c, z_m], dim=0)
        z_enc = torch.relu(self.ae_enc(z))
        z_rec = self.ae_dec(z_enc)

        return pred, z, z_rec

In [10]:
def hybrid_loss(pred, y, z, z_rec):
    # Compute class weight
    pos_weight = (len(y) - y.sum()) / (y.sum() + 1e-6)

    # Manual weighting
    weights = torch.where(y == 1, pos_weight, 1.0)

    bce = F.binary_cross_entropy(pred, y, weight=weights)

    mse = F.mse_loss(z_rec, z)

    return bce + 0.1 * mse

In [12]:
model = HybridGNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

for epoch in range(40):
    model.train()
    optimizer.zero_grad()

    pred, z, z_rec = model(data)

    loss = hybrid_loss(
        pred[train_mask],
        y[train_mask],
        z,
        z_rec
    )

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 2.2119
Epoch 5, Loss: 1.4611
Epoch 10, Loss: 1.3644
Epoch 15, Loss: 1.3310
Epoch 20, Loss: 1.2899
Epoch 25, Loss: 1.2795
Epoch 30, Loss: 1.2765
Epoch 35, Loss: 1.2713


In [16]:
model.eval()
with torch.no_grad():
    pred, _, _ = model(data)

y_true = y[test_mask].numpy()

best_f1 = 0
best_t = 0

for t in [0.3, 0.4, 0.5, 0.6]:
    pred_label = (pred[test_mask] > t).int().numpy()

    p = precision_score(y_true, pred_label)
    r = recall_score(y_true, pred_label)
    f1 = f1_score(y_true, pred_label)

    print(f"T={t} → P:{p:.3f} R:{r:.3f} F1:{f1:.3f}")

    if f1 > best_f1:
        best_f1 = f1
        best_t = t

print("\nBest Threshold:", best_t)

T=0.3 → P:0.176 R:1.000 F1:0.299
T=0.4 → P:0.182 R:0.975 F1:0.307
T=0.5 → P:0.179 R:0.843 F1:0.296
T=0.6 → P:0.180 R:0.667 F1:0.284

Best Threshold: 0.4


In [17]:
pred_label = (pred[test_mask] > best_t).int().numpy()

print("Accuracy:", accuracy_score(y_true, pred_label))
print("Precision:", precision_score(y_true, pred_label))
print("Recall:", recall_score(y_true, pred_label))
print("F1 Score:", f1_score(y_true, pred_label))
print("ROC-AUC:", roc_auc_score(y_true, pred[test_mask].numpy()))

Accuracy: 0.2276243093922652
Precision: 0.18235294117647058
Recall: 0.9748427672955975
F1 Score: 0.3072348860257681
ROC-AUC: 0.4856087814254641


In [26]:
# Pick any row index from dataframe
row_index = 150

row = df.iloc[row_index]

# Map to graph IDs
cust_id = cust_map[row['cc_num']]
merch_id = merch_map[row['merchant']]

edges = data['customer','transacts','merchant'].edge_index

# Find matching edge
match = ((edges[0] == cust_id) & (edges[1] == merch_id)).nonzero()

if len(match) > 0:
    idx = match[0].item()

    prob = pred[idx].item()
    predicted = 1 if prob > best_t else 0

    print("\n🔎 Specific Transaction Check")
    print("Amount:", row['amt'])
    print("Distance:", row['distance'])

    print("Probability:", prob)
    print("Predicted:", "FRAUD" if predicted==1 else "NOT FRAUD")
    print("Actual:", "FRAUD" if row['is_fraud']==1 else "NOT FRAUD")

    if predicted == row['is_fraud']:
        print("✅ Correct")
    else:
        print("❌ Wrong")
else:
    print("⚠️ Transaction not found in graph")


🔎 Specific Transaction Check
Amount: 0.0004983497651042584
Distance: 0.3544241183019
Probability: 0.44588521122932434
Predicted: FRAUD
Actual: FRAUD
✅ Correct
